In [2]:
include("../LiPoSID.jl")
using QuantumOptics
basis = NLevelBasis(2)
using LinearAlgebra
using HDF5
using DynamicPolynomials
using Dates
using TSSOS

## Non-Markovian system identification — full Kossakowski form

We fit the model:
$$\dot\rho(t) = \mathcal{L}_{\mathrm{Kos}}(H, C)\,\rho(t) + \mathcal{L}_{\mathrm{Kos}}(H, C_m)\,\rho(t - k\Delta t)$$

where the Kossakowski super-operator is
$$\mathcal{L}_{\mathrm{Kos}}(H, C)\,\rho = -i[H,\rho] + \frac{1}{2}\sum_{i,j=1}^{3} C_{ij}\bigl(2 f_i \rho f_j^\dagger - \rho f_j^\dagger f_i - f_j^\dagger f_i \rho\bigr)$$

with $\{f_1, f_2, f_3\} = \{\sigma_x/2, \sigma_y/2, \sigma_z/2\}$ the standard Kossakowski ONB.

Both super-operators share the **same Hamiltonian** $H$ but have independent Kossakowski matrices $C$ (current) and $C_m$ (memory), each constrained to be positive semidefinite.

Free variables:
- $H$: $\epsilon,\, h_{\mathrm{Re}},\, h_{\mathrm{Im}}$ (3 parameters)
- $C$: $\gamma_1, \gamma_2, \gamma_3, a_1, a_2, a_3$ (6 parameters)
- $C_m$: $\gamma_{m1}, \gamma_{m2}, \gamma_{m3}, a_{m1}, a_{m2}, a_{m3}$ (6 parameters)

**Constraints** (Sylvester criterion for $C \succeq 0$ and $C_m \succeq 0$): the three principal minors of each Kossakowski matrix must be non-negative.

In [3]:
# Kossakowski orthonormal basis: f_i = σ_i / 2
# satisfies tr(fᵢ fⱼ) = δᵢⱼ/2 and tr(fᵢ) = 0
σˣ = [ 0 1
       1 0 ]
σʸ = [ 0.   im
      -im   0. ]
σᶻ = [ 1.  0
       0  -1. ]

fᴷ₁ = σˣ / 2
fᴷ₂ = σʸ / 2
fᴷ₃ = σᶻ / 2

@assert tr(fᴷ₁ * fᴷ₂) ≈ 0  &&  tr(fᴷ₁ * fᴷ₃) ≈ 0  &&  tr(fᴷ₂ * fᴷ₃) ≈ 0
@assert tr(fᴷ₁ * fᴷ₁) ≈ 1/2  &&  tr(fᴷ₂ * fᴷ₂) ≈ 1/2  &&  tr(fᴷ₃ * fᴷ₃) ≈ 1/2

fᴷᴼᴺᴮ = [fᴷ₁, fᴷ₂, fᴷ₃]

3-element Vector{Matrix{ComplexF64}}:
 [0.0 + 0.0im 0.5 + 0.0im; 0.5 + 0.0im 0.0 + 0.0im]
 [0.0 + 0.0im 0.0 + 0.5im; 0.0 - 0.5im 0.0 + 0.0im]
 [0.5 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im -0.5 + 0.0im]

In [4]:
"""Kossakowski RHS (no control): -i[H,ρ] + (1/2) ΣᵢⱼCᵢⱼ(2fᵢρfⱼ† - ρfⱼ†fᵢ - fⱼ†fᵢρ)"""
function kossak_rhs(ρ, H, C, Fᴼᴺᴮ)
    U = (H * ρ - ρ * H) / im
    D = sum(C .* [2*fᵢ*ρ*fⱼ' - ρ*fⱼ'*fᵢ - fⱼ'*fᵢ*ρ for fᵢ in Fᴼᴺᴮ, fⱼ in Fᴼᴺᴮ]) / 2
    return U + D
end

kossak_rhs

In [5]:
"""Non-Markovian Kossakowski objective using Simpson quadrature.
Current-time term uses H, C; memory term (k steps back) uses same H, Cm."""
function non_mark_kossak_obj(ρ, t, H, C, Hm, Cm, k)
    obj = 0
    for i in (k+3):length(ρ)
        ρ_avg = (ρ[i-2]   + 4ρ[i-1]   + ρ[i])   / 3
        ρ_mem = (ρ[i-k-2] + 4ρ[i-k-1] + ρ[i-k]) / 3
        residual = ρ[i] - ρ[i-2] -
                   (t[i]-t[i-1]) * kossak_rhs(ρ_avg, H,  C,  fᴷᴼᴺᴮ) -
                   (t[i]-t[i-1]) * kossak_rhs(ρ_mem, Hm, Cm, fᴷᴼᴺᴮ)
        obj += LiPoSID.frobenius_norm2(residual)
    end
    if isempty(monomials(obj))
        obj = 0.
    else
        obj = sum(real(coef) * mon for (coef, mon) in zip(coefficients(obj), monomials(obj)))
    end
    return obj
end

non_mark_kossak_obj

In [6]:
"""Simulate non-Markovian Kossakowski dynamics with RK4.
Memory uses simulated history k steps back; for i ≤ k the initial state ρ[1] is used."""
function simulate_nonmark_kossak_rk4(ρ₀, t, H, C, Hm, Cm, k)
    ρ = Vector{Matrix{ComplexF64}}(undef, length(t))
    ρ[1] = ρ₀
    for i in 2:length(t)
        Δt    = t[i] - t[i-1]
        i_mem = max(1, i - k)
        ρ_mem = ρ[i_mem]
        Lm    = kossak_rhs(ρ_mem, Hm, Cm, fᴷᴼᴺᴮ)
        f(ρ_c) = kossak_rhs(ρ_c, H, C, fᴷᴼᴺᴮ) + Lm
        k1 = f(ρ[i-1])
        k2 = f(ρ[i-1] + Δt/2 * k1)
        k3 = f(ρ[i-1] + Δt/2 * k2)
        k4 = f(ρ[i-1] + Δt     * k3)
        ρ[i] = ρ[i-1] + Δt/6 * (k1 + 2k2 + 2k3 + k4)
    end
    return ρ
end

simulate_nonmark_kossak_rk4

In [7]:
"""Simulate non-Markovian Kossakowski dynamics using ODE solver (Tsit5, fixed step).
Memory uses simulated history k steps back; for i ≤ k the initial state ρ[1] is used."""
function simulate_nonmark_kossak(ρ₀, t, H, C, Hm, Cm, k)
    ρ = Vector{Matrix{ComplexF64}}(undef, length(t))
    ρ[1] = ρ₀
    for i in 2:length(t)
        Δt    = t[i] - t[i-1]
        i_mem = max(1, i - k)
        Lm    = kossak_rhs(ρ[i_mem], Hm, Cm, fᴷᴼᴺᴮ)
        function matrix_ode!(dρ, ρ_c, p, t)
            dρ .= kossak_rhs(ρ_c, H, C, fᴷᴼᴺᴮ) + Lm
        end
        prob = ODEProblem(matrix_ode!, ρ[i-1], (t[i-1], t[i]))
        sol  = solve(prob, dt=Δt, adaptive=false)
        ρ[i] = sol.u[end]
    end
    return ρ
end

simulate_nonmark_kossak

In [8]:
"""Constrained sparse SOS optimisation via TSSOS.
Minimises obj subject to constr[i] ≥ 0 for all i.
Returns (solution, optimal_value, status)."""
function cs_tssos_opt(obj::Polynomial, constr::Vector{<:Polynomial})
    all_vars = union(variables(obj), reduce(union, variables.(constr)))
    solution   = variables(obj) => nothing
    opt_val    = nothing
    status     = :Error
    relax_order = maxdegree(obj) > 2 ? maxdegree(obj) : 2
    try
        redirect_stdout(open("/dev/null", "w")) do
            redirect_stderr(open("/dev/null", "w")) do
                opt, sol, data = TSSOS.cs_tssos_first(
                    [obj; constr...], all_vars, relax_order;
                    numeq=0, solution=true, QUIET=true)
                prev_opt, prev_sol, prev_data = opt, sol, data
                while sol !== nothing
                    prev_opt, prev_sol, prev_data = opt, sol, data
                    opt, sol, data = TSSOS.cs_tssos_higher!(data; QUIET=true, solution=true)
                end
                ref_sol, flag = TSSOS.refine_sol(prev_opt, prev_sol, prev_data;
                                                 QUIET=true, tol=1e-10)
                solution = variables(obj) => ref_sol
                opt_val  = prev_opt
                status   = flag == 0 ? :Global : :Local
            end
        end
    catch e
        println("TSSOS failed: ", e)
    end
    return solution, opt_val, status
end

cs_tssos_opt

### Symbolic setup

Hamiltonian (shared between current and memory terms):
$$H = \frac{1}{2}\begin{pmatrix} \epsilon & h_{\mathrm{Re}} + i h_{\mathrm{Im}} \\ h_{\mathrm{Re}} - i h_{\mathrm{Im}} & -\epsilon \end{pmatrix}$$

Kossakowski matrix parameterisation (Theorem 3.1, GKS 1976):
$$C = \begin{pmatrix} -\gamma_1+\gamma_2+\gamma_3 & -ia_3 & ia_2 \\ ia_3 & \gamma_1-\gamma_2+\gamma_3 & -ia_1 \\ -ia_2 & ia_1 & \gamma_1+\gamma_2-\gamma_3 \end{pmatrix}$$

$C \succeq 0$ iff the Sylvester minors $\kappa_i \geq 0$,  $\kappa_1\kappa_2+\kappa_2\kappa_3+\kappa_3\kappa_1 \geq a_1^2+a_2^2+a_3^2$, and $\det C \geq 0$, where $\kappa_i$ are the diagonal entries.

In [9]:
# --- Hamiltonian (shared) ---
@polyvar ϵ h_Re h_Im

H0ˢʸᵐᵇ = [ ϵ                h_Re + im*h_Im
            h_Re - im*h_Im  -ϵ              ] / 2

# --- Current Kossakowski matrix C ---
@polyvar γ[1:3] a[1:3]

κ₁ = -γ[1]+γ[2]+γ[3]
κ₂ =  γ[1]-γ[2]+γ[3]
κ₃ =  γ[1]+γ[2]-γ[3]

Cˢʸᵐᵇ = [ κ₁       -im*a[3]   im*a[2]
           im*a[3]   κ₂       -im*a[1]
          -im*a[2]   im*a[1]   κ₃      ]

# PSD constraints for C
c_constr1 = γ[1] + γ[2] + γ[3]                                          # tr(C)/2 ≥ 0
c_constr2 = κ₁*κ₂ + κ₂*κ₃ + κ₃*κ₁ - a[1]^2 - a[2]^2 - a[3]^2         # sum of 2×2 minors ≥ 0
c_constr3 = κ₁*κ₂*κ₃ - κ₁*a[1]^2 - κ₂*a[2]^2 - κ₃*a[3]^2             # det(C) ≥ 0

# --- Memory Kossakowski matrix Cm ---
@polyvar γm[1:3] am[1:3]

κm₁ = -γm[1]+γm[2]+γm[3]
κm₂ =  γm[1]-γm[2]+γm[3]
κm₃ =  γm[1]+γm[2]-γm[3]

Cmˢʸᵐᵇ = [ κm₁        -im*am[3]    im*am[2]
            im*am[3]    κm₂        -im*am[1]
           -im*am[2]    im*am[1]    κm₃      ]

# PSD constraints for Cm
cm_constr1 = γm[1] + γm[2] + γm[3]
cm_constr2 = κm₁*κm₂ + κm₂*κm₃ + κm₃*κm₁ - am[1]^2 - am[2]^2 - am[3]^2
cm_constr3 = κm₁*κm₂*κm₃ - κm₁*am[1]^2 - κm₂*am[2]^2 - κm₃*am[3]^2

constraints = [γ[1], γ[2], γ[3], c_constr1, c_constr2, c_constr3]
#               ,γm[1], γm[2], γm[3], cm_constr1, cm_constr2, cm_constr3]

6-element Vector{Polynomial{true, Int64}}:
 γ₁
 γ₂
 γ₃
 γ₁ + γ₂ + γ₃
 -γ₁² + 2γ₁γ₂ + 2γ₁γ₃ - γ₂² + 2γ₂γ₃ - γ₃² - a₁² - a₂² - a₃²
 -γ₁³ + γ₁²γ₂ + γ₁²γ₃ + γ₁γ₂² - 2γ₁γ₂γ₃ + γ₁γ₃² + γ₁a₁² - γ₁a₂² - γ₁a₃² - γ₂³ + γ₂²γ₃ + γ₂γ₃² - γ₂a₁² + γ₂a₂² - γ₂a₃² - γ₃³ - γ₃a₁² - γ₃a₂² + γ₃a₃²

In [10]:
data_dir   = "../DATA/"
models_dir = "../MODELS/"
tests_dir  = "../TESTS/"

dodeca_files = ["State_D"*string(n) for n=1:20]
basis_files  = ["State_B"*string(n) for n=1:4]

train_files = basis_files
test_files  = dodeca_files

k = 1  # memory steps back (τ = k·Δt)

1

In [13]:
function g_nonmark_kossak_objective(γᵢ)
    obj = 0
    for df_trn in train_files
        ρᵗʳⁿ, tᵗʳⁿ = LiPoSID.get_rho_series(data_dir*df_trn*"_2CUT_data.h5", γᵢ)
        end_train   = min(length(tᵗʳⁿ), 1200)
        ρᵗʳⁿ = convert(Vector{Matrix{ComplexF64}}, ρᵗʳⁿ[1:end_train])
        tᵗʳⁿ = convert(Vector{Float64},            tᵗʳⁿ[1:end_train])
        #obj += non_mark_kossak_obj(ρᵗʳⁿ, tᵗʳⁿ, H0ˢʸᵐᵇ, Cˢʸᵐᵇ, H0ˢʸᵐᵇ, Cmˢʸᵐᵇ, k)
        obj += non_mark_kossak_obj(ρᵗʳⁿ, tᵗʳⁿ, H0ˢʸᵐᵇ, Cˢʸᵐᵇ, zeros(2,2), zeros(3,3), k)
    end
    return obj
end

g_nonmark_kossak_objective (generic function with 1 method)

In [12]:
using DifferentialEquations

In [14]:
γ = ["0.079477", "0.25133", "0.79477", "2.5133", "7.9477", "25.133", "79.477", "251.33"]

date_and_time_string = string(Dates.format(now(), "yyyy-u-dd_at_HH-MM"))
tests_data_file_name = "NonMark_MarkTST_Kossak_k$(k)_trn4_tst20_"*date_and_time_string*".h5"

subs_num(sym, sol) = begin v = subs(sym, sol); try; convert(Float64, v); catch; 0.0; end; end
subs_mat(mat, sol) = map(mat) do e; v = subs(e, sol); try; convert(Float64, v)+0im; catch; 0.0+0im; end; end
subs_cmat(mat, sol) = map(mat) do e; v = subs(e, sol); try; convert(ComplexF64, v); catch; 0.0+0im; end; end

"""Project matrix to nearest valid density matrix: Hermitian, PSD, trace-1."""
function to_density_op(basis, ρ)
    ρh = Hermitian((ρ + ρ') / 2)
    F  = eigen(ρh)
    vals = max.(real(F.values), 0.0)
    s = sum(vals)
    s > 0 && (vals ./= s)
    DenseOperator(basis, Matrix(Hermitian(F.vectors * Diagonal(vals) * F.vectors')))
end

for γᵢ in γ
    println("γ = "*γᵢ)

    poly = g_nonmark_kossak_objective(γᵢ)
    sol, opt_val, status = cs_tssos_opt(poly, constraints)
    println("  status: ", status, "  opt_val: ", opt_val)

    Hˢⁱᵈ  = subs_cmat(H0ˢʸᵐᵇ,  sol)
    Cˢⁱᵈ  = subs_cmat(Cˢʸᵐᵇ,   sol)
    Cmˢⁱᵈ = subs_cmat(Cmˢʸᵐᵇ,  sol)

    epsilon = subs_num(ϵ, sol)
    println("  ε=$(epsilon)")

    h5open(tests_dir*tests_data_file_name, "cw") do fid
        γ_group = create_group(fid, "gamma_"*γᵢ)
        γ_group["H"]    = Hˢⁱᵈ
        γ_group["C"]    = Cˢⁱᵈ
        γ_group["Cm"]   = Cmˢⁱᵈ
        γ_group["k"]    = k
        γ_group["epsilon"] = epsilon
    end

    for df in test_files
        print(" "*df*"-")

        ρₛ, tₛ = LiPoSID.get_rho_series(data_dir*df*"_2CUT_data.h5", γᵢ)
        ρₛ = convert(Vector{Matrix{ComplexF64}}, ρₛ)
        tₛ = convert(Vector{Float64}, tₛ)

        #ρˢⁱᵈ = simulate_nonmark_kossak_rk4(ρₛ[1], tₛ, Hˢⁱᵈ, Cˢⁱᵈ, Hˢⁱᵈ, Cmˢⁱᵈ, k)
        #ρˢⁱᵈ = simulate_nonmark_kossak(ρₛ[1], tₛ, Hˢⁱᵈ, Cˢⁱᵈ, Hˢⁱᵈ, Cmˢⁱᵈ, k)
        ρˢⁱᵈ = simulate_nonmark_kossak(ρₛ[1], tₛ, Hˢⁱᵈ, Cˢⁱᵈ, zeros(2,2), zeros(3,3), k)
        
        ρᵗˢᵗ    = [DenseOperator(basis, Hermitian(ρₜ))  for ρₜ in ρₛ]
        ρˢⁱᵈ_op = [to_density_op(basis, ρₜ)             for ρₜ in ρˢⁱᵈ]

        Fₑₓ = [abs(fidelity(ρ₁, ρ₂)) for (ρ₁, ρ₂) in zip(ρᵗˢᵗ, ρˢⁱᵈ_op)]

        h5open(tests_dir*tests_data_file_name, "cw") do fid
            γ_group          = open_group(fid, "gamma_"*γᵢ)
            init_state_group = create_group(γ_group, df)
            init_state_group["Fidelity"] = convert.(Float64, Fₑₓ)
        end

        print(round(minimum(Fₑₓ), sigdigits=3)) 
    end

    println()
end

println(tests_data_file_name)

γ = 0.079477
  status: Local  opt_val: 0.00019919239815491358
  ε=25.126144390535927
 State_D1-0.989 State_D2-0.989 State_D3-0.998 State_D4-0.998 State_D5-0.993 State_D6-0.993 State_D7-0.996 State_D8-0.996 State_D9-0.983 State_D10-1.0 State_D11-0.998 State_D12-0.998 State_D13-0.989 State_D14-0.989 State_D15-0.998 State_D16-0.998 State_D17-0.996 State_D18-0.996 State_D19-1.0 State_D20-0.983
γ = 0.25133
  status: Local  opt_val: 0.00020715536720979536
  ε=25.140971309824486
 State_D1-0.983 State_D2-0.983 State_D3-0.985 State_D4-0.985 State_D5-0.983 State_D6-0.983 State_D7-0.984 State_D8-0.984 State_D9-0.983 State_D10-0.986 State_D11-0.985 State_D12-0.985 State_D13-0.983 State_D14-0.983 State_D15-0.985 State_D16-0.985 State_D17-0.984 State_D18-0.984 State_D19-0.986 State_D20-0.983
γ = 0.79477
  status: Local  opt_val: 0.00027567327428446216
  ε=25.16295511621526
 State_D1-0.998 State_D2-0.998 State_D3-0.999 State_D4-0.999 State_D5-0.998 State_D6-0.998 State_D7-0.998 State_D8-0.998 State_D